In [1]:
import ijson
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
INPUT_FILE = r"D:\AHRC\Irrigation_Git\lag_adjustment\final_output\all_predictions.json"
OUTPUT_FILE = r"D:\AHRC\Irrigation_Git\lag_adjustment\final_output\data.parquet"

In [3]:
CHUNK_SIZE = 50_000
writer = None
chunk = []

with open(INPUT_FILE, "rb") as f:
    for record in ijson.items(f, "item"):
        chunk.append(record)
        if len(chunk) >= CHUNK_SIZE:
            df = pd.DataFrame(chunk)
            table = pa.Table.from_pandas(df)
            if writer is None:
                writer = pq.ParquetWriter(OUTPUT_FILE, table.schema)
            writer.write_table(table)
            chunk = []

if chunk:
    df = pd.DataFrame(chunk)
    table = pa.Table.from_pandas(df)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_FILE, table.schema)
    writer.write_table(table)

if writer:
    writer.close()

print(f"Done! Saved to {OUTPUT_FILE}")

SystemError: <class 'decimal.Decimal'> returned a result with an exception set

In [ ]:
from pathlib import Path
import ijson
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
INPUT_FILE = r"D:\AHRC\Irrigation_Git\lag_adjustment\final_output\all_predictions.json"
OUTPUT_FILE = r"D:\AHRC\Irrigation_Git\lag_adjustment\final_output\data.parquet"
ARRAY_CHUNK = 500_000

In [ ]:
schema = pa.schema([
    ("idx", pa.int64()),
    ("y_true", pa.float64()),
    ("y_pred", pa.float64()),
])
OUT_DIR = Path(OUTPUT_FILE)


In [ ]:
def write_array_chunks(model, cv_type, fold, split_name, y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    if len(y_true) != len(y_pred):
        raise ValueError(
            f"Length mismatch for model={model} fold={fold} split={split_name}"
        )

    part_dir = (
        OUT_DIR
        / f"model={model}"
        / f"cv_type={cv_type}"
        / f"fold={int(fold)}"
        / f"split={split_name}"
    )
    part_dir.mkdir(parents=True, exist_ok=True)

    total = len(y_true)
    for start in range(0, total, ARRAY_CHUNK):
        end = min(start + ARRAY_CHUNK, total)

        table = pa.Table.from_pydict(
            {
                "idx": np.arange(start, end, dtype=np.int64),
                "y_true": np.asarray(y_true[start:end], dtype=np.float64),
                "y_pred": np.asarray(y_pred[start:end], dtype=np.float64),
            },
            schema=schema,
        )

        pq.write_table(
            table,
            part_dir / f"part_{start:012d}_{end:012d}.parquet",
            compression="zstd",
        )

with open(INPUT_FILE, "rb") as f:
    # Assumes your JSON file is a top-level array of pred_record objects
    for rec in ijson.items(f, "item", use_float=True):
        model = rec["model"]
        cv_type = rec["cv_type"]
        fold = rec["fold"]

        write_array_chunks(
            model, cv_type, fold, "test",
            rec["y_test"], rec["y_pred_test"]
        )
        write_array_chunks(
            model, cv_type, fold, "train",
            rec["y_train"], rec["y_pred_train"]
        )

print(f"Done! Partitioned parquet files written under: {OUT_DIR}")
